In [0]:
dbutils.widgets.dropdown("environment", "dev", ["dev", "preprod", "prod"], "Environment")

In [0]:
%run /Workspace/Users/lakshmisas96@gmail.com/orders-project/orders-analytics-pipeline/utils/config_loader

In [0]:
config = load_config_from_widget()
print(f"🚀 Bronze Orders Pipeline - {config['environment'].upper()} Environment")
print("Status: Running bronze ingestion...")

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, when

orders_path = config['storage']['landing']['orders']
print(f"📂 Reading from: {orders_path}")

df_orders = (spark.read.format("parquet").load(orders_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("environment", lit(config['environment']))
    .withColumn("order_priority",
        when(col("total_amount") < 100, "Low")
        .when(col("total_amount") < 500, "Medium")
        .when(col("total_amount") < 1000, "High")
        .otherwise("Urgent")
    )
)

bronze_schema = config['schemas']['bronze']
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['catalog']}.{bronze_schema}")

external_path = f"{config['storage']['external']['bronze']}/orders_raw"
print(f"📂 Writing to: {external_path}")

bronze_table = f"{config['catalog']}.{bronze_schema}.orders_raw"

df_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", external_path) \
    .saveAsTable(bronze_table)

print(f"✅ {bronze_table}: {spark.table(bronze_table).count()} records")
print(f"✅ Delta logs created at: {external_path}/_delta_log/")